In [ ]:
import pandas as pd
import os
os.environ["LOKY_MAX_CPU_COUNT"] = "16"

In [ ]:
data=pd.read_csv("watson_healthcare_modified.csv")

In [ ]:
data.head()

In [ ]:
data.isnull().sum()

In [ ]:
data.duplicated().sum()

In [ ]:
data.info()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
# To check class imbalance
sns.countplot(x=data['Attrition'])
# showing the class imbalance

In [ ]:
# checking the BusinessTravel category 
sns.countplot(x=data['BusinessTravel'])

In [ ]:
plt.pie(data['Department'].value_counts(),labels=['Maternity','Cardiology','Neurology'],autopct='%1.1f%%')

In [ ]:
sns.catplot(x=data['EducationField'],y=data['Age'])
plt.xticks(rotation=45)

In [ ]:
# we also check the age distrubution
sns.histplot(data['Age'])

In [ ]:
sns.kdeplot(data['Age'])
# we can say data is not too much right skweed its skweed but not more
print(data['Age'].skew())
print(data['Age'].median())
print(data['Age'].mean())
# not too much mean is grater ok so data is best not large skew

In [ ]:
sns.lineplot(x=data['YearsSinceLastPromotion'],y=data['YearsAtCompany'])

In [ ]:
# now we delete the contant value  features(over all value in column is same and there is no change so this not use ful for model)
print(data['Over18'].value_counts())
print(data['EmployeeCount'].value_counts())
print(data['StandardHours'].value_counts())

In [ ]:
cleaned_var_data=data.drop(columns=['Over18','EmployeeCount','StandardHours'])

In [ ]:
# Also clear the EmploymentId it also not affect on model means no relation of employee id and our training
cleaned_var_data=cleaned_var_data.drop('EmployeeID',axis=1)

In [ ]:
# now we can use heatmap also for relation best visualization
plt.figure(figsize=(20,10))
sns.heatmap(cleaned_var_data.select_dtypes('number').corr(),cmap='RdBu',annot=True)

In [ ]:
# from above we notice one thing that there is multicolinearity between joblevel and monthly income so to avoid model confusion we remove one of them
# cleaned_var_data=cleaned_var_data.drop('JobLevel',axis=1)
cleaned_data=cleaned_var_data

In [ ]:
# now we do one hot and label encoding for our categorical data
from sklearn.preprocessing import OneHotEncoder,LabelEncoder
lbl=LabelEncoder()
cleaned_data['Attrition']=lbl.fit_transform(cleaned_var_data['Attrition'])
cleaned_data['OverTime']=lbl.fit_transform(cleaned_var_data['OverTime'])
cleaned_data['Gender']=lbl.fit_transform(cleaned_var_data['Gender'])

In [ ]:
# one hot encoding
ohe=OneHotEncoder(drop='first',sparse_output=True)
embed=ohe.fit_transform(cleaned_data[['BusinessTravel','Department','EducationField','JobRole','MaritalStatus']])
df=pd.DataFrame(embed.toarray(),columns=ohe.get_feature_names_out(),index=cleaned_data.index)
fresh_data=pd.concat([cleaned_data,df],axis=1)

In [ ]:
fresh_data=fresh_data.drop(columns=['BusinessTravel','Department','EducationField','JobRole','MaritalStatus'])

In [ ]:
# now my data is ready for model
fresh_data.info()

In [ ]:
X=fresh_data.drop('Attrition',axis=1)
y=fresh_data['Attrition']

In [ ]:
# spliting data
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test=train_test_split(X,y,test_size=0.2,stratify=y)

In [ ]:
# now feature scaling
from sklearn.preprocessing import StandardScaler
scl=StandardScaler()
x_train_scaled=scl.fit_transform(x_train)
x_test_scaled=scl.transform(x_test)

In [ ]:
# Logistic regression
from sklearn.linear_model import LogisticRegression
lg=LogisticRegression(class_weight='balanced')
lg.fit(x_train_scaled,y_train)

In [ ]:
logistic_prediction=lg.predict(x_test_scaled)

In [ ]:
from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score,roc_auc_score,confusion_matrix

In [ ]:
print('LogisticRegression accuracy_score',accuracy_score(y_test,logistic_prediction))
print('LogisticRegression precision_score',precision_score(y_test,logistic_prediction))
print('LogisticRegression recall_score',recall_score(y_test,logistic_prediction))
print('LogisticRegression f1_score',f1_score(y_test,logistic_prediction))
print('LogisticRegression roc_auc_score',roc_auc_score(y_test,logistic_prediction))
print('LogisticRegression confusion_matrix\n',confusion_matrix(y_test,logistic_prediction))

In [ ]:
# its good result because we have to focus on recall 
# lg-good

In [ ]:
# KNN
from sklearn.neighbors import KNeighborsClassifier
knn=KNeighborsClassifier(n_neighbors=5)
knn.fit(x_train_scaled,y_train)

In [ ]:
knn_prediction=knn.predict(x_test_scaled)

In [ ]:
print('KNeighborsClassifier accuracy_score',accuracy_score(y_test,knn_prediction))
print('KNeighborsClassifier precision_score',precision_score(y_test,knn_prediction))
print('KNeighborsClassifier recall_score',recall_score(y_test,knn_prediction))
print('KNeighborsClassifier f1_score',f1_score(y_test,knn_prediction))
print('KNeighborsClassifier roc_auc_score',roc_auc_score(y_test,knn_prediction))
print('KNeighborsClassifier confusion_matrix\n',confusion_matrix(y_test,knn_prediction))

In [ ]:
# because data has 40 feature its work poorly the sparsecity is come here
# knn-is bad for this

In [ ]:
# Naive bayes

In [ ]:
from sklearn.naive_bayes import GaussianNB
naive=GaussianNB()
naive.fit(x_train_scaled,y_train)

In [ ]:
naive_prediction=naive.predict(x_test_scaled)

In [ ]:
print('naive_bayes accuracy_score',accuracy_score(y_test,naive_prediction))
print('naive_bayes precision_score',precision_score(y_test,naive_prediction))
print('naive_bayes recall_score',recall_score(y_test,naive_prediction))
print('naive_bayes f1_score',f1_score(y_test,naive_prediction))
print('naive_bayes roc_auc_score',roc_auc_score(y_test,naive_prediction))
print('naive_bayes confusion_matrix\n',confusion_matrix(y_test,naive_prediction))

In [ ]:
# this also not gives much better performance but recall is best


In [ ]:
from sklearn.tree import DecisionTreeClassifier
dtc=DecisionTreeClassifier(max_depth=4,class_weight='balanced')
dtc.fit(x_train_scaled,y_train)

In [ ]:
dtc_prediction=dtc.predict(x_test_scaled)

In [ ]:
print('dtc accuracy_score',accuracy_score(y_test,dtc_prediction))
print('dtc precision_score',precision_score(y_test,dtc_prediction))
print('dtc recall_score',recall_score(y_test,dtc_prediction))
print('dtc f1_score',f1_score(y_test,dtc_prediction))
print('dtc roc_auc_score',roc_auc_score(y_test,dtc_prediction))
print('dtc confusion_matrix\n',confusion_matrix(y_test,dtc_prediction))

In [ ]:
dtc_prediction1=dtc.predict(x_train_scaled)
print('dtc accuracy_score',accuracy_score(y_train,dtc_prediction1))
print('dtc precision_score',precision_score(y_train,dtc_prediction1))
print('dtc recall_score',recall_score(y_train,dtc_prediction1))
print('dtc f1_score',f1_score(y_train,dtc_prediction1))
print('dtc roc_auc_score',roc_auc_score(y_train,dtc_prediction1))
print('dtc confusion_matrix\n',confusion_matrix(y_train,dtc_prediction1))

In [ ]:
# this is also give a best recall and moderate accuracy

In [ ]:
from sklearn.ensemble import RandomForestClassifier
rnd=RandomForestClassifier(class_weight='balanced',n_estimators=201,min_samples_split=60,ccp_alpha=0.03,random_state=42)
rnd.fit(x_train_scaled,y_train)

In [ ]:
rnd_predict=rnd.predict(x_test_scaled)

In [ ]:
print('rnd accuracy_score',accuracy_score(y_test,rnd_predict))
print('rnd precision_score',precision_score(y_test,rnd_predict))
print('rnd recall_score',recall_score(y_test,rnd_predict))
print('rnd f1_score',f1_score(y_test,rnd_predict))
print('rnd roc_auc_score',roc_auc_score(y_test,rnd_predict))
print('rnd confusion_matrix\n',confusion_matrix(y_test,rnd_predict))

In [ ]:
dtc_prediction1=rnd.predict(x_train_scaled)
print('dtc accuracy_score',accuracy_score(y_train,dtc_prediction1))
print('dtc precision_score',precision_score(y_train,dtc_prediction1))
print('dtc recall_score',recall_score(y_train,dtc_prediction1))
print('dtc f1_score',f1_score(y_train,dtc_prediction1))
print('dtc roc_auc_score',roc_auc_score(y_train,dtc_prediction1))
print('dtc confusion_matrix\n',confusion_matrix(y_train,dtc_prediction1))

In [ ]:
# this also the not good for this data sets

In [ ]:
from xgboost import XGBClassifier

In [ ]:
xgb=XGBClassifier()
xgb.fit(x_train_scaled,y_train)

In [ ]:
xgbpredict=xgb.predict(x_test_scaled)

In [ ]:
print('xgbpredict accuracy_score',accuracy_score(y_test,xgbpredict))
print('xgbpredict precision_score',precision_score(y_test,xgbpredict))
print('xgbpredict recall_score',recall_score(y_test,xgbpredict))
print('xgbpredict f1_score',f1_score(y_test,xgbpredict))
print('xgbpredict roc_auc_score',roc_auc_score(y_test,xgbpredict))
print('xgbpredict confusion_matrix\n',confusion_matrix(y_test,xgbpredict))

In [ ]:
# it also not good bad recall

In [ ]:
from sklearn.ensemble import StackingClassifier
from sklearn.ensemble import AdaBoostClassifier
dtc=DecisionTreeClassifier()
base_model=[('a',AdaBoostClassifier()),('r',RandomForestClassifier(n_estimators=301)),('g',GaussianNB())]
l=LogisticRegression(class_weight='balanced')
stc=StackingClassifier(estimators=base_model,final_estimator=l)
stc.fit(x_train_scaled,y_train)

In [ ]:
stc_predict=stc.predict(x_test_scaled)

In [ ]:
print('StackingClassifier accuracy_score',accuracy_score(y_test,stc_predict))
print('StackingClassifier precision_score',precision_score(y_test,stc_predict))
print('StackingClassifier recall_score',recall_score(y_test,stc_predict))
print('StackingClassifier f1_score',f1_score(y_test,stc_predict))
print('StackingClassifier roc_auc_score',roc_auc_score(y_test,stc_predict))
print('StackingClassifier confusion_matrix\n',confusion_matrix(y_test,stc_predict))

In [ ]:
# comparing the all classification algorithms on imbalance data and finding the best recall that the need done

In [ ]:
# now we can also visualize the all recall value of all model
stackingclassifier=recall_score(y_test,stc_predict)
xgboost=recall_score(y_test,xgbpredict)
random_forest=recall_score(y_test,rnd_predict)
decisiontree=recall_score(y_test,dtc_prediction)
naive=recall_score(y_test,naive_prediction)
knn=recall_score(y_test,knn_prediction)
logistic=recall_score(y_test,logistic_prediction)
list1=[stackingclassifier,xgboost,random_forest,decisiontree,naive,knn,logistic]
list2=['Decision Tree','Logistic Regression','Naive Bayes','stacking Classifier','Random Forest','xgboost','knn']
list1.sort()
list1.reverse()

In [ ]:
plt.bar(list2,list1)
plt.tight_layout()
plt.xticks(rotation=45)
plt.ylabel('recall')
plt.xlabel('Classification Algorithms')
plt.title('Classification Algorithms and Recall')

In [ ]:
decisiontree